In [ ]:
# Download historical stock data for SBI
import yfinance as yf
df = yf.Ticker("SBIN.NS").history(period="max")
df.to_csv("SBI_historical_data.csv")
df.head()

# Plot the closing price over time
import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
plt.plot(df['Close'], label='Close Price', color='blue')
plt.title('SBI Stock Close Price Over Time')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

# Calculate moving averages and their ratios
def moving_average_and_ratio_close(data):
    data['MA_week'] = data['Close'].rolling(window=5).mean().shift(1)
    data['MA_month'] = data['Close'].rolling(window=21).mean().shift(1)
    data['MA_year'] = data['Close'].rolling(window=252).mean().shift(1)
    data['MA_ratio_week_month'] = data['MA_week'] / data['MA_month']
    data['MA_ratio_week_year'] = data['MA_week'] / data['MA_year']
    data['MA_ratio_month_year'] = data['MA_month'] / data['MA_year']
    return data

df = moving_average_and_ratio_close(df)
print(df.tail(5))

In [ ]:
# Import necessary libraries
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

# Initialize SGD Regressor
sgd = SGDRegressor(loss='squared_error', penalty='l2', max_iter=1000, alpha=0.1, learning_rate='constant', eta0=0.01, random_state=42)

# Prepare the data
df = df.dropna(axis=0)
X = df.drop(columns=['Close', 'Dividends', 'Stock Splits'], axis=1)
Y = df['Close']
X_train = X[:int(0.8*len(X))]
X_test = X[int(0.8*len(X)):]
Y_train = Y[:int(0.8*len(Y))]
Y_test = Y[int(0.8*len(Y)):]

# Train SGD model
sgd.fit(X_train, Y_train)
prediction = sgd.predict(X_test)

# Calculate error
print('SGD error:', mean_squared_error(Y_test, prediction))

# Plot predictions
import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
plt.plot(Y_test.values, label='Actual Close Prices', color='blue')
plt.plot(prediction, label='Predicted Close Prices', color='red')
plt.title('SBI Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()
plt.show()

# TensorFlow Neural Network Model
import tensorflow as tf
layer0 = tf.keras.layers.Dense(units=1, input_shape=[X_train.shape[1]])
model = tf.keras.Sequential([layer0])

# Compile and train the model
model.compile(optimizer='adam', loss='mse')
model.fit(X_train, Y_train, epochs=50, batch_size=32, validation_split=0.2)

# Predict and calculate error
prediction = model.predict(X_test)
print('Neural Network error:', mean_squared_error(Y_test, prediction))

# Plot Neural Network predictions
plt.figure(figsize=(12,6))
plt.plot(Y_test.values, label='Actual Close Prices', color='blue')
plt.plot(prediction, label='Predicted Close Prices', color='red')
plt.title('Actual vs Predicted Close Prices')
plt.xlabel('Time')
plt.ylabel('Close Price')
plt.legend()
plt.show()

# Decision Tree Regressor
from sklearn.tree import DecisionTreeRegressor
regressor = DecisionTreeRegressor(max_depth=10, min_samples_split=3, random_state=42)
regressor.fit(X_train, Y_train)

# Predict and calculate error
predictions = regressor.predict(X_test)
print('Decision Tree error:', mean_squared_error(Y_test, predictions))

# Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=3, random_state=42)
regressor.fit(X_train, Y_train)
predictions = regressor.predict(X_test)
print('Random Forest error:', mean_squared_error(Y_test, predictions))

# Grid Search with Time Series Cross Validation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0], 'eta0': [0.001, 0.01, 0.1]}
tscv = TimeSeriesSplit(n_splits=5)
grid_search = GridSearchCV(sgd, param_grid, cv=tscv, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, Y_train)
print("Best parameters:", grid_search.best_params_)
best_sgd = grid_search.best_estimator_
predictions_sgd = best_sgd.predict(X_test_scaled)
print('Grid Search SGD error:', mean_squared_error(Y_test, predictions_sgd))